##  Section B: RAG Fundamentals, Implementation



## Corpus: Rohit Sharma


In [2]:
corpus = """
Rohit Sharma is an Indian international cricketer who has represented India in all three formats of cricket.
He is mainly known for his batting and has been one of India's important players in limited-overs cricket.
His calm batting style and ability to score big runs have made him a well-known player in international cricket.

Rohit Sharma has played an important role in Indian Premier League cricket.
He has played for Mumbai Indians and has also served as the captain of the team for several seasons.
Under his captaincy, Mumbai Indians achieved multiple IPL titles and became one of the successful teams in the league.

In One Day International cricket, Rohit Sharma is known for his ability to score large individual totals.
He has scored multiple double centuries in ODI cricket and has also produced several important innings for India.
His batting performances have made him one of the notable opening batsmen in modern ODI cricket.

Rohit has also represented India in several T20 World Cup tournaments.
He was part of the Indian team that won the T20 World Cup in 2024.
His experience and leadership were important parts of India's limited-overs cricket during this period.

Rohit Sharma has received several awards and achievements during his cricket career.
He has performed consistently in international cricket and has contributed to India's success in important tournaments.
His performances in both international cricket and the IPL have made him a recognized name in Indian cricket.

Apart from batting, Rohit Sharma is also known for his leadership skills.
His experience as an international player and IPL captain has helped him make important decisions on the cricket field.
He has played an important role as a senior member of the Indian cricket team.
"""


In [13]:

def chunk_with_overlap(text, size=20, overlap=10):
    """Split text into overlapping chunks."""

    words = text.split()
    chunks = []
    start = 0

    while start < len(words):
        end = start + size
        chunks.append(" ".join(words[start:end]))

        if end >= len(words):
            break

        start += size - overlap

    return chunks

# Create chunks
chunks = chunk_with_overlap(corpus, size=20, overlap=10)

# Print the resulting chunks
for i, chunk in enumerate(chunks, 1):
    print(f"Chunk {i}:")
    print(chunk)
    print("-" * 80)

Chunk 1:
Rohit Sharma is an Indian international cricketer who has represented India in all three formats of cricket. He is mainly
--------------------------------------------------------------------------------
Chunk 2:
India in all three formats of cricket. He is mainly known for his batting and has been one of India's
--------------------------------------------------------------------------------
Chunk 3:
known for his batting and has been one of India's important players in limited-overs cricket. His calm batting style and
--------------------------------------------------------------------------------
Chunk 4:
important players in limited-overs cricket. His calm batting style and ability to score big runs have made him a well-known
--------------------------------------------------------------------------------
Chunk 5:
ability to score big runs have made him a well-known player in international cricket. Rohit Sharma has played an important
---------------------------------------

**Source : https://www.bitavox.com/chapters/29**

### Explanation

The corpus is divided into chunks of 20 words.


*   An overlap of 10 words is maintained between consecutive chunks.

*   This means that the last 10 words of one chunk are also included in the next chunk.
*   This helps preserve the context of the information and reduces the chance of losing important information when a sentence or topic is divided between two chunks.



*   The corpus is divided into smaller chunks of 20 words, which keeps each chunk short and focused on a particular piece of information.










# B2: Dense retrieval


In [14]:
import numpy as np
from sentence_transformers import SentenceTransformer

# Load embedding model
embedder = SentenceTransformer("all-MiniLM-L6-v2")

# Embed all chunks
chunk_vecs = embedder.encode(chunks, normalize_embeddings=True)

# Dense retrieval function
def retrieve(query, k=3):
    """Return the k chunks most similar to the query."""

    q = embedder.encode(query, normalize_embeddings=True)

    # Cosine similarity
    scores = chunk_vecs @ q

    # Get top-k results
    top = np.argsort(-scores)[:k]

    return [(chunks[i], scores[i]) for i in top]


# Test Query 1
query1 = "What are Rohit Sharma's achievements in ODI cricket?"

print("Query 1:", query1)
results = retrieve(query1, k=3)

for chunk, score in results:
    print(f"Score: {score:.4f}")
    print(chunk)
    print("-" * 80)

# Test Query 2
query2 = "Which IPL team has Rohit Sharma captained?"

print("Query 2:", query2)
results = retrieve(query2, k=3)

for chunk, score in results:
    print(f"Score: {score:.4f}")
    print(chunk)
    print("-" * 80)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Query 1: What are Rohit Sharma's achievements in ODI cricket?
Score: 0.7691
limited-overs cricket during this period. Rohit Sharma has received several awards and achievements during his cricket career. He has performed
--------------------------------------------------------------------------------
Score: 0.7582
ability to score big runs have made him a well-known player in international cricket. Rohit Sharma has played an important
--------------------------------------------------------------------------------
Score: 0.7538
cricket. Apart from batting, Rohit Sharma is also known for his leadership skills. His experience as an international player and
--------------------------------------------------------------------------------
Query 2: Which IPL team has Rohit Sharma captained?
Score: 0.7703
the IPL have made him a recognized name in Indian cricket. Apart from batting, Rohit Sharma is also known for
--------------------------------------------------------------------------------


# B3: Sparse retrieval (BM25)


In [16]:
!pip install rank-bm25

In [17]:
from rank_bm25 import BM25Okapi
import numpy as np

# Tokenize the chunks
tokenized_chunks = [chunk.split() for chunk in chunks]

# Create BM25 index
bm25 = BM25Okapi(tokenized_chunks)


# BM25 retrieval function
def bm25_retrieve(query, k=3):
    """Return the k most relevant chunks using BM25."""

    tokenized_query = query.split()

    scores = bm25.get_scores(tokenized_query)

    top = np.argsort(scores)[::-1][:k]

    return [(chunks[i], scores[i]) for i in top]

# Test Query 1
query1 = "What are Rohit Sharma's achievements in ODI cricket?"

print("Query 1:", query1)
results = bm25_retrieve(query1, k=3)

for chunk, score in results:
    print(f"Score: {score:.4f}")
    print(chunk)
    print("-" * 80)


# Test Query 2
query2 = "Which IPL team has Rohit Sharma captained?"

print("Query 2:", query2)
results = bm25_retrieve(query2, k=3)

for chunk, score in results:
    print(f"Score: {score:.4f}")
    print(chunk)
    print("-" * 80)

Query 1: What are Rohit Sharma's achievements in ODI cricket?
Score: 2.8236
awards and achievements during his cricket career. He has performed consistently in international cricket and has contributed to India's success
--------------------------------------------------------------------------------
Score: 2.7791
batsmen in modern ODI cricket. Rohit has also represented India in several T20 World Cup tournaments. He was part of
--------------------------------------------------------------------------------
Score: 2.7740
limited-overs cricket during this period. Rohit Sharma has received several awards and achievements during his cricket career. He has performed
--------------------------------------------------------------------------------
Query 2: Which IPL team has Rohit Sharma captained?
Score: 2.9292
the team for several seasons. Under his captaincy, Mumbai Indians achieved multiple IPL titles and became one of the successful
-----------------------------------------------------

**Source -https://pypi.org/project/rank-bm25/?**

### Comparison of Dense Retrieval and BM25

- Dense retrieval uses embeddings and cosine similarity to find chunks based on their meaning.
- BM25 uses keyword matching, so it works well when the query contains important words that also appear in the chunks.
- For Query 1, dense retrieval can find relevant ODI-related information even when the wording of the query is different from the text.
- For Query 2, BM25 can work well because words such as "IPL", "team", and "captain" directly match the words present in the chunks.
- Therefore, the better method depends on the query: dense retrieval is useful for semantic meaning, while BM25 is useful for direct keyword matching.